# 04 — Analyse results

Loads the CSVs written by notebook 03 and renders the cross-scenario tables and plots used in Chapters 5 and 6 of the thesis.

Headline metrics include started vs. completed charging, % agents who waited / charged, median / 95th-percentile waiting time, total queue-minutes, max queue length, and a normalised **objective score** (lower = better).

## What is being optimised

From the perspective of a city government, the planning problem is **where to allocate scarce public charging capacity in central Warsaw**. The simulation does not solve a closed-form optimisation — instead it estimates, for any candidate station layout, the four signals a planner would minimise:

$$ J = \alpha\,W + \beta\,D + \gamma\,Q + \delta\,U_{imb} $$

* **W** — waiting pressure on affected drivers (mean waiting time among agents who actually waited)
* **D** — induced detour distance (mean per-agent detour, metres)
* **Q** — cumulative queue pressure (total queue-minutes across all stations)
* **U_imb** — utilisation imbalance across stations (standard deviation of utilisation)

Each component is min-max normalised across the three scenarios in a single run, so $J\!\in\![0,1]$ and **lower is better**. Default weights are equal ($\alpha=\beta=\gamma=\delta=0.25$); they can be changed in `src/config.py → ObjectiveWeights`. The thesis still reports the four raw components separately — $J$ is a **decision-support summary**, not a claim of true social welfare.

The model **compares scenarios under identical demand** rather than searching over all possible layouts: real road geometry, real charger inventory, fixed RNG seed, fixed agent population. That keeps the comparison interpretable and reproducible for thesis chapters 5–6.

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve().parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

import pandas as pd
import matplotlib.pyplot as plt
from src import config

In [ ]:
summary = pd.read_csv(config.TABLES_DIR / 'scenario_summary.csv')
agents = pd.read_csv(config.TABLES_DIR / 'agent_results.csv')
stations = pd.read_csv(config.TABLES_DIR / 'station_results.csv')
professor = pd.read_csv(config.TABLES_DIR / 'professor_summary.csv')
summary.T  # transpose so all metrics fit vertically

### Compact decision-support table (saved to outputs/tables/professor_summary.csv)

In [ ]:
professor

### Per-scenario aggregates (agent-level)

In [ ]:
agents.groupby('scenario').agg(
    completed_trips=('completed_trips', 'sum'),
    started_charging=('started_charging_events', 'sum'),
    completed_charging=('completed_charging_events', 'sum'),
    times_queued=('times_queued', 'sum'),
    mean_wait=('waiting_time_min', 'mean'),
    median_wait=('waiting_time_min', 'median'),
    p95_wait=('waiting_time_min', lambda s: s.quantile(0.95)),
    max_wait=('waiting_time_min', 'max'),
    mean_detour_m=('detour_distance_m', 'mean'),
    median_detour_m=('detour_distance_m', 'median'),
    stranded=('stranded', 'sum'),
)

### Waiters only — the meaningful waiting-time view
Use explicit quantile aggregation so we are not exposed to differences in the columns returned by `pandas.DataFrame.describe()` across versions.

In [ ]:
waiters = agents[agents['waiting_time_min'] > 0]
waiters.groupby('scenario')['waiting_time_min'].agg(
    n='count',
    mean='mean',
    median='median',
    p95=lambda s: s.quantile(0.95),
    max='max',
)

### Per-scenario aggregates (station-level)

In [ ]:
stations.groupby('scenario').agg(
    n_stations=('station_id', 'count'),
    total_ports=('n_ports', 'sum'),
    mean_util=('utilisation', 'mean'),
    sd_util=('utilisation', lambda s: s.std(ddof=0)),
    max_util=('utilisation', 'max'),
    total_queue_min=('total_queue_minutes', 'sum'),
    max_queue=('max_queue', 'max'),
    total_started=('total_started', 'sum'),
    total_completed=('total_completed', 'sum'),
)

## Key preliminary figures

**Figure 1, `professor_queue_over_time.png`**, shows total queue length across all stations over the simulation horizon. It directly represents the queue-pressure component of the planning objective.

**Figure 2, `professor_objective_components.png`**, compares the normalised objective components across scenarios. Lower values indicate better performance. The figure summarises waiting pressure, detour distance, queue pressure, and utilisation imbalance in a compact decision-support view.

**Preliminary interpretation.** The simulation estimates the components of the planning objective for each candidate layout. The **distributed** layout produces the lowest queue pressure, lower waiting among affected agents, and the shortest mean detour distance. The **clustered** layout creates the highest queue accumulation and longest detours despite using the same total number of stations and ports — the same capacity, concentrated in one place, fails to absorb spatially distributed demand. The **real** OCM/EIPA layout sits between the two on most components.

In [ ]:
from IPython.display import Image, display
for name in [
    'professor_queue_over_time.png',
    'professor_objective_components.png',
    'professor_waiters_only.png',
]:
    p = config.FIGURES_DIR / name
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f'missing: {p}')

### Other figures (rendered by notebook 03)

In [ ]:
from IPython.display import Image, display
for name in [
    'queue_over_time.png',
    'scenario_comparison_waiting_time.png',
    'waiting_time_waiters_only.png',
    'detour_distribution.png',
    'station_utilisation.png',
    'utilisation_histogram.png',
    'charging_events_by_scenario.png',
]:
    p = config.FIGURES_DIR / name
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f'missing: {p}')